In [42]:
import pandas as pd
from scipy.stats import f_oneway, chisquare, tukey_hsd, ttest_ind
import os
import numpy as np

In [ ]:
def categorical_patients_variables(df):
    grouped_df = df.groupby("ID").first() 
    number_of_patients = len(df.index.unique())

    # SEX
    count_sex = grouped_df["SEX"].value_counts()
    men_frequency = 100 * count_sex.iloc[0] / number_of_patients if len(count_sex) > 0 else 0

    # ONSET SYMPTOMS
    spinal_counts_sum = 0
    other_counts_sum = 0
    bulbar_counts_sum = 0
    for idx, count in grouped_df["ONSET_SYMPTOMS"].value_counts().items():
        if idx.endswith("SPINAL"):
            spinal_counts_sum += count
        elif idx == "BULBAR":
            bulbar_counts_sum += count
        elif idx in ["SPINAL_AND_BULBAR", "RESPIRATORY", "COGNITIVE", "OTHER"]:
            other_counts_sum += count
    ni_counts_sum = number_of_patients - (spinal_counts_sum + bulbar_counts_sum + other_counts_sum)

    # ARM
    if "ARM" in grouped_df.columns:
        observational_counts_sum = 0
        treated_counts_sum = 0
        placebo_counts_sum = 0
        for idx, count in grouped_df["ARM"].value_counts().items():
            if idx in ["Case", "ALS"]:
                observational_counts_sum += count
            elif idx in ["Active", "TRO19622"]:
                treated_counts_sum += count
            elif idx == "Placebo":
                placebo_counts_sum += count
    else:
        observational_counts_sum = treated_counts_sum = placebo_counts_sum = 0

    return {
        'number_of_patients': number_of_patients,
        "men_count": count_sex.iloc[0],
        "men_freq": men_frequency.round(1),
        "spinal_count": spinal_counts_sum,
        "spinal_freq": round(100 * spinal_counts_sum/ number_of_patients, 1),
        'bulbar_count': bulbar_counts_sum,
        "bulbar_freq": round(100 * bulbar_counts_sum/ number_of_patients,1),
        'other_count': other_counts_sum,
        "other_freq": round(100 * other_counts_sum/ number_of_patients,1),
        'ni_count': ni_counts_sum,
        "ni_freq": round(100 * ni_counts_sum/ number_of_patients,1),
        "observational_freq": round(100 * observational_counts_sum/ number_of_patients,1),
        'observational_count': observational_counts_sum,
        "treated_freq": round(100 * treated_counts_sum/ number_of_patients,1),
        'treated_count': treated_counts_sum,
        "placebo_freq": round(100 * placebo_counts_sum/ number_of_patients,1),
        'placebo_count': placebo_counts_sum,
    }

def comparison_df(dict_columns, categorical_res):
    df = pd.DataFrame(index=dict_rows.keys(), columns=list(dict_columns.keys()) + ['p value'])

    # Mapping for relationships: PROACT-PULSE → 'a', etc.
    dataset_names = list(dict_columns.keys())
    relation_labels = {}
    label_counter = ord('a')
    for i in range(len(dataset_names)):
        for j in range(i + 1, len(dataset_names)):
            pair = f"{dataset_names[i]}-{dataset_names[j]}"
            relation_labels[pair] = chr(label_counter)
            label_counter += 1
    print(relation_labels)

    for column, id_df in dict_columns.items():
        df.loc['Number of patients', column] = categorical_res[column]["number_of_patients"]
        
        for row, id in dict_rows.items():
            if row in continuous_cols:
                if id in id_df.columns:
                    grouped = id_df.groupby("ID")[id].first().dropna()
                    if not grouped.empty:
                        mean_value = grouped.mean().round(1)
                        std_value = grouped.std().round(1)
                        df.loc[row, column] = f"{mean_value:.1f} ± {std_value:.1f}"
                    else:
                        df.loc[row, column] = "-"
                else:
                    df.loc[row, column] = "-"
            elif row in categorical_cols:
                count = id + '_count'
                freq = id + '_freq'
                if categorical_res[column][count] == 0:
                    df.loc[row, column] = '-'
                else:
                    df.loc[row, column] = f"{categorical_res[column][count]} ({categorical_res[column][freq]}%)"

    # Add resting patients from PROACT to observational
    if "PROACT" in dict_columns.keys():
        proact_obs_arm_count = (categorical_res["PROACT"]["number_of_patients"]
                                - categorical_res["PROACT"].get("treated_count", 0)
                                - categorical_res["PROACT"].get("placebo_count", 0))
        df.loc['Arm - Observational', "PROACT"] = f"{proact_obs_arm_count} ({round(100 * proact_obs_arm_count / categorical_res['PROACT']['number_of_patients'], 1)}%)"

    # Add p value for categorical
    for categorical_row in ['Male', 'Onset - Spinal', "Onset - Bulbar"]:
        categorical_groups = [categorical_res[dataset][dict_rows[categorical_row] + '_freq'] for dataset in categorical_res]
        pval = chisquare(categorical_groups)[1]
        df.loc[categorical_row, 'p value'] = "< 0.0001" if pval < 0.0001 else round(pval, 4)

    # Add p value for continuous
    for continuous_row in ['Age at First symptoms', "Age at Diagnosis", 'Age at Baseline', 'ALSFRS-r', 'BMI', 'FVC(%)','SVC(%)',"MMT",  'HHD Average', "BP Diast", "BP Syst", "Respiration Rate", "Age at First Gastrostomy", "Age at First VNI", "Age at First Tracheostomy", "Age at Death"]:
    #for continuous_row in ['Age at First symptoms', "Age at Diagnosis", 'Age at Baseline', 'ALSFRS-r', 'BMI']:
        valid_groups = []
        valid_names = []
        for name, dataset in dict_columns.items():
            var = dict_rows[continuous_row]
            if var in dataset.columns:
                values = dataset.groupby("ID")[var].first().dropna().values
                if len(values) > 0:
                    valid_groups.append(values)
                    valid_names.append(name)
        
        if len(valid_groups) >= 2:
            # T-test when only 2 cohorts to compare
            if len(valid_groups) == 2:
                pval = ttest_ind(*valid_groups)[1]
            # ANOVA when more than 2 cohorts to compare
            else:
                pval = f_oneway(*valid_groups)[1]
            pval_str = "< 0.0001" if pval < 0.0001 else round(pval, 4).astype(str)

            # Tukey only for continuous value as there is no pval < 0.05 on categorical variables
            significant_letters = []
            if pval < 0.05 and len(valid_groups) > 2:
                tukey_result = tukey_hsd(*valid_groups)
                for i in range(len(valid_groups)):
                    for j in range(i + 1, len(valid_groups)):
                        p = tukey_result.pvalue[i, j]
                        if p < 0.05:
                            label = relation_labels.get(f"{valid_names[i]}-{valid_names[j]}") or relation_labels.get(f"{valid_names[j]}-{valid_names[i]}")
                            if label:
                                significant_letters.append(label)
            if significant_letters:
                pval_str += f" ({''.join(significant_letters)})"
            df.loc[continuous_row, 'p value'] = pval_str
            
    df[df.isna()] = "-"
    return df


In [44]:
df_proact_leaspy_ready = pd.read_csv("../_data/leaspy_ready/PROACT_leaspy_ready.csv").set_index(["ID"])
df_pulse_leaspy_ready = pd.read_csv("../_data/leaspy_ready/PULSE_leaspy_ready.csv").set_index(["ID"])
df_answerals_leaspy_ready = pd.read_csv("../_data/leaspy_ready/ANSWERALS_leaspy_ready.csv").set_index(["ID"])
df_trophos_leaspy_ready = pd.read_csv("../_data/leaspy_ready/TROPHOS_leaspy_ready.csv").set_index(["ID"])
df_neurobank_leaspy_ready = pd.read_csv("../_data/leaspy_ready/NEUROBANK_leaspy_ready.csv").set_index(["ID"])

## Basic cohorts comparison at baseline


In [ ]:
datasets = {
    "PROACT": df_proact_leaspy_ready,
    "PULSE": df_pulse_leaspy_ready,
    "ANSWERALS": df_answerals_leaspy_ready,
    "TROPHOS": df_trophos_leaspy_ready,
    "NEUROBANK": df_neurobank_leaspy_ready,
}

results = {name: categorical_patients_variables(df) for name, df in datasets.items()}

In [48]:
dict_rows = {
    "Number of patients": 'number_of_patients',
    "Male": 'men',
    "Age at First symptoms": "AGE_AT_FIRST_SYMPTOMS",
    "Age at Diagnosis": "AGE_AT_DIAGNOSIS", 
    "Age at Baseline": "AGE_AT_BASELINE", 
    "Onset - Spinal": 'spinal', 
    "Onset - Bulbar": 'bulbar', 
    "Onset - Other": 'other', 
    "Onset - N.I.": 'ni',
    "Arm - Active": 'treated', 
    "Arm - Placebo": 'placebo', 
    "Arm - Observational": "observational",
    "ALSFRS-r": "ALSFRS_R_TOTAL",
    "BMI": 'BMI',
    "FVC(%)": 'PERCENT_FORCED_VITAL_CAPACITY_GLI_METHOD_AVERAGE',
    "SVC(%)": 'PERCENT_SLOW_VITAL_CAPACITY_GLI_METHOD_AVERAGE',
    "MMT": "MANUAL_MUSCLE_TESTING_SUM",
    "HHD Average": "HHD_AVERAGE_TOTAL",
    "BP Diast": "BLOOD_PRESSURE_DIASTOLIC",
    "BP Syst": "BLOOD_PRESSURE_SYSTOLIC",
    "Respiration Rate": "RESPIRATION_RATE",
    "Age at First Gastrostomy": "AGE_AT_FIRST_GASTROSTOMY",
    "Age at First VNI": "AGE_AT_FIRST_VNI",
    "Age at First Tracheostomy": "AGE_AT_FIRST_TRACHEOTOMY",
    "Age at Death": "AGE_AT_DEATH",
}

categorical_cols = ["Male", "Onset - Spinal",
                    "Onset - Bulbar", "Onset - Other", "Onset - N.I.",
                    "Arm - Active",  "Arm - Placebo", "Arm - Observational"]

continuous_cols = ["Age at First symptoms","Age at Diagnosis","Age at Baseline", 
                   "ALSFRS-r","BMI","FVC(%)","SVC(%)", "MMT",  'HHD Average', "BP Diast", "BP Syst", "Respiration Rate", 
                   "Age at First Gastrostomy", "Age at First VNI", "Age at First Tracheostomy", "Age at Death"]

In [49]:
df = comparison_df(datasets, results)

{'PROACT-PULSE': 'a', 'PROACT-ANSWERALS': 'b', 'PROACT-TROPHOS': 'c', 'PROACT-NEUROBANK': 'd', 'PULSE-ANSWERALS': 'e', 'PULSE-TROPHOS': 'f', 'PULSE-NEUROBANK': 'g', 'ANSWERALS-TROPHOS': 'h', 'ANSWERALS-NEUROBANK': 'i', 'TROPHOS-NEUROBANK': 'j'}


In [50]:
df

,PROACT,PULSE,ANSWERALS,TROPHOS,NEUROBANK,p value
Number of patients,10106,497,850,511,2620,-
Male,6259 (61.9%),305 (61.4%),532 (62.6%),330 (64.6%),1473 (56.2%),0.9591
Age at First symptoms,53.8 ± 11.5,62.1 ± 11.1,57.1 ± 11.5,55.2 ± 11.2,61.0 ± 11.9,< 0.0001 (abdefhij)
Age at Diagnosis,54.6 ± 11.6,63.2 ± 11.1,58.5 ± 11.4,55.8 ± 11.2,62.5 ± 11.7,< 0.0001 (abdefhij)
Age at Baseline,56.5 ± 11.5,63.5 ± 11.1,59.8 ± 11.1,56.6 ± 11.2,63.9 ± 11.3,< 0.0001 (abdefhij)
Onset - Spinal,6506 (64.4%),363 (73.0%),614 (72.2%),410 (80.2%),1767 (67.4%),0.7287
Onset - Bulbar,1776 (17.6%),126 (25.4%),182 (21.4%),101 (19.8%),615 (23.5%),0.7849
Onset - Other,446 (4.4%),7 (1.4%),44 (5.2%),-,87 (3.3%),-
Onset - N.I.,1378 (13.6%),1 (0.2%),10 (1.2%),-,151 (5.8%),-
Arm - Active,5318 (52.6%),-,-,259 (50.7%),-,-


- a: PROACT - PULSE
- b: PROACT - ANSWERALS
- c: PROACT - TROPHOS
- d: PROACT - NEUROBANK 
- e: PULSE - ANSWERALS
- f: PULSE - TROPHOS
- g: PULSE - NEUROBANK
- h: ANSWERALS - TROPHOS
- i: ANSWERALS - NEUROBANK
- j : TROPHOS - NEUROBANK